In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from pathlib import Path

current_dir = Path.cwd()

if current_dir.name == "notebooks":
    project_dir = current_dir.parent
else:
    project_dir = current_dir

video_path = project_dir / "data" / "synthetic_videos" / "synthetic_circle.mp4"

output_video_path = project_dir / "outputs" / "processed_videos" / "lucas_kanade_synthetic.mp4"
output_csv_path = project_dir / "outputs" / "plots" / "lucas_kanade_points.csv"

output_video_path.parent.mkdir(parents=True, exist_ok=True)
output_csv_path.parent.mkdir(parents=True, exist_ok=True)

print("Project directory:", project_dir)
print("Video path:", video_path)
print("Video exists:", video_path.exists())

In [ ]:
def track_motion_with_lucas_kanade(
    input_video_path: Path,
    output_video_path: Path,
):
    """
    Track visual feature points using Lucas-Kanade optical flow.
    """

    cap = cv2.VideoCapture(str(input_video_path))

    if not cap.isOpened():
        raise RuntimeError("Could not open the input video")

    fps = cap.get(cv2.CAP_PROP_FPS)
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

    fourcc = cv2.VideoWriter_fourcc(*"mp4v")
    writer = cv2.VideoWriter(str(output_video_path), fourcc, fps, (width, height))

    ret, old_frame = cap.read()

    if not ret:
        raise RuntimeError("Could not read the first frame")

    old_gray = cv2.cvtColor(old_frame, cv2.COLOR_BGR2GRAY)

    feature_params = dict(
        maxCorners=100,
        qualityLevel=0.2,
        minDistance=7,
        blockSize=7
    )

    lk_params = dict(
        winSize=(15, 15),
        maxLevel=2,
        criteria=(
            cv2.TERM_CRITERIA_EPS | cv2.TERM_CRITERIA_COUNT,
            10,
            0.03
        )
    )

    p0 = cv2.goodFeaturesToTrack(old_gray, mask=None, **feature_params)

    if p0 is None:
        raise RuntimeError("No feature points found in the first frame")

    mask = np.zeros_like(old_frame)

    tracked_data = []
    frame_index = 0

    while True:
        ret, frame = cap.read()

        if not ret:
            break

        frame_gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)

        p1, status, error = cv2.calcOpticalFlowPyrLK(
            old_gray,
            frame_gray,
            p0,
            None,
            **lk_params
        )

        if p1 is None:
            break

        good_new = p1[status == 1]
        good_old = p0[status == 1]

        displacements = []

        for new_point, old_point in zip(good_new, good_old):
            x_new, y_new = new_point.ravel()
            x_old, y_old = old_point.ravel()

            dx = x_new - x_old
            dy = y_new - y_old
            displacement = np.sqrt(dx**2 + dy**2)
            displacements.append(displacement)

            tracked_data.append({
                "frame": frame_index,
                "x_old": x_old,
                "y_old": y_old,
                "x_new": x_new,
                "y_new": y_new,
                "dx": dx,
                "dy": dy,
                "displacement": displacement
            })

            x_new_i, y_new_i = int(x_new), int(y_new)
            x_old_i, y_old_i = int(x_old), int(y_old)

            mask = cv2.line(
                mask,
                (x_new_i, y_new_i),
                (x_old_i, y_old_i),
                (255, 0, 0),
                2
            )

            frame = cv2.circle(
                frame,
                (x_new_i, y_new_i),
                4,
                (0, 0, 255),
                -1
            )

        output_frame = cv2.add(frame, mask)

        if len(displacements) > 0:
            mean_displacement = np.mean(displacements)
        else:
            mean_displacement = 0

        cv2.putText(
            output_frame,
            f"Frame: {frame_index} | LK points: {len(good_new)}",
            (20, 30),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.7,
            (255, 255, 255),
            2
        )

        cv2.putText(
            output_frame,
            f"Mean displacement: {mean_displacement:.2f} px/frame",
            (20, 60),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.7,
            (255, 255, 255),
            2
        )

        writer.write(output_frame)

        old_gray = frame_gray.copy()
        p0 = good_new.reshape(-1, 1, 2)

        frame_index += 1

    cap.release()
    writer.release()

    tracked_df = pd.DataFrame(tracked_data)

    return tracked_df

In [ ]:
tracked_df = track_motion_with_lucas_kanade(
    input_video_path=video_path,
    output_video_path=output_video_path
)

tracked_df.to_csv(output_csv_path, index=False)

print("Lucas-Kanade video saved at:", output_video_path)
print("Tracked points saved at:", output_csv_path)
print("Number of tracked point records:", len(tracked_df))

tracked_df.head()